In [ ]:
import cv2
import numpy as np
import random

# 设置视频的一些基本参数，长、宽、帧数、总时长
frame_width = 480
frame_height = 380
fps = 10
duration = 60
total_frames = fps * duration

# 定义视频的编码格式和输出文件
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('moving_numbers_rotated.avi', fourcc, fps, (frame_width, frame_height), True)

# 创建一个类表示移动的数字
class MovingNumber:
    def __init__(self):
        # 随机生成一个0-9的数字
        self.number = random.randint(0, 9)
        # 以下初始位置、速度、加速度、颜色、字体大小、旋转角度和速度，均在给定范围内随机生成。
        self.position = np.array([random.randint(0, frame_width), random.randint(0, frame_height)], dtype=np.float32)
        self.velocity = np.array([random.uniform(-2, 2), random.uniform(-2, 2)], dtype=np.float32)
        self.acceleration = np.array([random.uniform(-0.1, 0.1), random.uniform(-0.1, 0.1)], dtype=np.float32)
        self.color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
        self.font_scale = random.uniform(1.0, 2.0)
        self.angle = random.uniform(-10, 10)
        self.rotation_velocity = random.uniform(-10, 10)

    # 定义一种方法，用来控制旋转和移动
    def move(self):
        # 更新速度和位置
        self.velocity += self.acceleration
        self.position += self.velocity

        # 更新旋转角度
        self.angle += self.rotation_velocity

        # 边界判断：判断数字是否还在屏幕范围内
        if (self.position[0] < 0 or self.position[0] > frame_width or
            self.position[1] < 0 or self.position[1] > frame_height):
            return False  # 如果不在范围内，就使其自动消失
        return True

# 创建10个移动数字类的实例
numbers = [MovingNumber() for _ in range(10)]

# for循环开始生成每一帧
for i in range(total_frames):
    # 纯色背景
    background = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
    # 为背景添加一些随机噪声
    noise = np.random.normal(0, 25, (frame_height, frame_width, 3)).astype(np.uint8)
    background += noise

    # 还在屏幕内的数字
    new_numbers = []
    for number in numbers:
        if number.move():
            # 创建一个空的图片，放置文本
            text_img = np.zeros_like(background, dtype=np.uint8)
            # 计算文本的大小，确定文本的绘制位置
            text_size = cv2.getTextSize(str(number.number), cv2.FONT_HERSHEY_SIMPLEX, number.font_scale, 2)[0]
            text_origin = (int(number.position[0] - text_size[0] // 2),
                           int(number.position[1] + text_size[1] // 2))

            # 在空图片上绘制数字文本
            cv2.putText(text_img, str(number.number), text_origin, cv2.FONT_HERSHEY_SIMPLEX,
                        number.font_scale, number.color, 2, cv2.LINE_AA)

            # 创建旋转矩阵并应用到文本图片上，以此实现旋转
            M = cv2.getRotationMatrix2D(tuple(number.position), number.angle, 1)
            rotated_text_img = cv2.warpAffine(text_img, M, (frame_width, frame_height))

            # 创建掩膜，并将旋转后的文本图片叠加到背景上
            mask = rotated_text_img > 0
            background[mask] = rotated_text_img[mask]

            # 将仍在屏幕内的数字保存到新列表
            new_numbers.append(number)

    # 只保留仍在屏幕内的数字
    numbers = new_numbers


    if random.random() < 0.1:
        numbers.append(MovingNumber())


    out.write(background)


out.release()
cv2.destroyAllWindows()
